In [ ]:
%pip install numpy mne osl-dynamics tensorflow ipyevents ipywidgets pyvista pyvistaqt pandas seaborn nibabel nilearn scikit-image numba pyyaml scikit-learn

In [2]:
import sys
!conda run -n hmmnet pip install h5io

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import mne
import numpy as np
import scipy.io
import h5py

from osl_dynamics import analysis, inference, files
from osl_dynamics.meeg import preproc, rhino, source_recon, parcellation
from osl_dynamics.data import Data
from osl_dynamics.utils import plotting
from osl_dynamics.utils.filenames import OSLFilenames


In [ ]:
# =============================================================================
# 1. LOAD DATA FROM MAT FILE
# =============================================================================

subject_folder = '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/'
subject_filename = 'Subject5_spm_mne'

filepath = subject_folder + subject_filename + '.mat'

with h5py.File(filepath, 'r') as f:
    data       = np.array(f['data']).T        # (n_channels, n_times)
    sfreq      = float(np.array(f['fs']).flatten()[0])
    chan_names = []
    for ref in f['chan_names'][:, 0]:
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        chan_names.append(name)
    print(f['sens'])



print(f"Data shape: {data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq} Hz")

Not setting metadata
14801 matching events found
No baseline correction applied
0 projection items activated
Error processing subject 5: name 'os' is not defined
Not setting metadata
14801 matching events found
No baseline correction applied
0 projection items activated
Error processing subject 6: name 'os' is not defined
Error processing subject 7: [Errno 2] Unable to synchronously open file (unable to open file: name = '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s07/Subject7_spm_mne.mat', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
Not setting metadata
14801 matching events found
No baseline correction applied
0 projection items activated
Error processing subject 8: name 'os' is not defined


: 

: 

In [ ]:
# =============================================================================
# 2. BUILD MNE EPOCHS OBJECT
# =============================================================================

# isolate EEG data from total channels
eeg_idx = [i for i, name in enumerate(chan_names) if name.startswith("EEG")]
eeg_names = [chan_names[i] for i in eeg_idx]
data_eeg = data[eeg_idx, :]

print(f"EEG data shape: {data_eeg.shape}")
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

# reorder data_eeg to have shape (n_epochs/trials, n_channels, n_times)#
data_eeg = data_eeg.transpose(1, 0, 2)
print("Reordered data_eeg shape:", data_eeg.shape)
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

# create MNE info object for EEG data
info = mne.create_info(
    ch_names=eeg_names,
    sfreq=sfreq,
    ch_types=["eeg"] * len(eeg_names),
)

# create MNE Epoch Array for EEG data (with shape (n_epochs, n_channels, n_times))
Epochs_data_array = mne.EpochsArray(data_eeg, info)
print(f"dev_head_t: {Epochs_data_array.info['dev_head_t']}")

ds directory : \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\mg4670_katja2_20160623_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
      -0.02   76.26    0.00 mm <->   -0.02   76.26   -0.00 mm (orig :  -47.05   64.14 -266.25 mm) diff =    0.000 mm
       0.02  -76.26    0.00 mm <->    0.02  -76.26    0.00 mm (orig :   55.90  -47.62 -279.35 mm) diff =    0.000 mm
      99.56    0.00    0.00 mm <->   99.56   -0.00   -0.00 mm (orig :   77.83   74.42 -260.68 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    4 EEG electrode locations assigned to channel info.
    4 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\mg4670_katja2_20160623_01.d

In [24]:
# =============================================================================
# 3. SAVE
# =============================================================================
import os

os.chdir(subject_folder)        
outfile = subject_folder + subject_filename + '.fif'
Epochs_data_array.save(outfile, overwrite=True)
print("Saved!")


C:\Users\chick\AppData\Local\Temp\ipykernel_33404\3061489649.py:8: RuntimeWarning: This filename (//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/Subject5_spm_mne.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  Epochs_data_array.save(outfile, overwrite=True)


Saved!
